In [6]:
# Create Corpus from the .jsonl file

import pandas as pd
import json

rows = []

with open("all.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        
        rows.append({
            "Question": obj.get("question"),
            "Human": obj.get("human_answers"),
            "ChatGPT_Old": obj.get("chatgpt_answers"),
            "Source": obj.get("source")
        })

chatGPT_old = pd.DataFrame(rows)

In [7]:
# Create question index for taking random sample for ChatGPT prompts

chatGPT_old = chatGPT_old.reset_index(drop=True)
chatGPT_old["Question_ID"] = chatGPT_old.index

chatGPT_old.head()

,Question,Human,ChatGPT_Old,Source,Question_ID
0,"Why is every book I hear about a "" NY Times # ...","[Basically there are many categories of "" Best...",[There are many different best seller lists th...,reddit_eli5,0
1,"If salt is so bad for cars , why do we use it ...",[salt is good for not dying in car crashes and...,[Salt is used on roads to help melt ice and sn...,reddit_eli5,1
2,Why do we still have SD TV channels when HD lo...,[The way it works is that old TV stations got ...,[There are a few reasons why we still have SD ...,reddit_eli5,2
3,Why has nobody assassinated Kim Jong - un He i...,[You ca n't just go around assassinating the l...,[It is generally not acceptable or ethical to ...,reddit_eli5,3
4,How was airplane technology able to advance so...,[Wanting to kill the shit out of Germans drive...,[After the Wright Brothers made the first powe...,reddit_eli5,4


In [14]:
# Create dataframe of new ChatGPT responses

chatGPT_new = pd.read_csv("chatgptNew_filled.csv")
chatGPT_new = chatGPT_new.drop(columns=["Question"])
chatGPT_new.head()

,Question_ID,ChatGPT_New
0,11458,Sloths aren’t “lazy” so much as perfectly buil...
1,1092,A simple way to think about it is that big com...
2,5193,The short answer is that these percentages are...
3,7288,The brain doesn’t really prefer even numbers o...
4,3198,"The American Dream is the idea that anyone, no..."


In [15]:
# Merge human, old ChatGPT, and new ChatGPT responses together

corpus = chatGPT_old.merge(chatGPT_new, on="Question_ID", how="left")
corpus.head()

,Question,Human,ChatGPT_Old,Source,Question_ID,ChatGPT_New
0,"Why is every book I hear about a "" NY Times # ...","[Basically there are many categories of "" Best...",[There are many different best seller lists th...,reddit_eli5,0,NaN
1,"If salt is so bad for cars , why do we use it ...",[salt is good for not dying in car crashes and...,[Salt is used on roads to help melt ice and sn...,reddit_eli5,1,NaN
2,Why do we still have SD TV channels when HD lo...,[The way it works is that old TV stations got ...,[There are a few reasons why we still have SD ...,reddit_eli5,2,NaN
3,Why has nobody assassinated Kim Jong - un He i...,[You ca n't just go around assassinating the l...,[It is generally not acceptable or ethical to ...,reddit_eli5,3,NaN
4,How was airplane technology able to advance so...,[Wanting to kill the shit out of Germans drive...,[After the Wright Brothers made the first powe...,reddit_eli5,4,NaN


In [16]:
# Build three aligned dataframes
human_df = corpus[["Question_ID", "Question", "Source", "Human"]].copy()
human_df = human_df.rename(columns={"Human": "Response"})
human_df["Group"] = "Human"

old_ai_df = corpus[["Question_ID", "Question", "Source", "ChatGPT_Old"]].copy()
old_ai_df = old_ai_df.rename(columns={"ChatGPT_Old": "Response"})
old_ai_df["Group"] = "Old_AI"

new_ai_df = corpus[["Question_ID", "Question", "Source", "ChatGPT_New"]].copy()
new_ai_df = new_ai_df.rename(columns={"ChatGPT_New": "Response"})
new_ai_df["Group"] = "New_AI"

# Stack them together
long_df = pd.concat([human_df, old_ai_df, new_ai_df], ignore_index=True)

# Cleanup
long_df = long_df[["Question_ID", "Question", "Source", "Group", "Response"]]
long_df = long_df.dropna(subset=["Response"])
long_df["Response"] = long_df["Response"].astype(str).str.strip()
long_df = long_df[long_df["Response"] != ""]

print(long_df.head())
print(long_df["Group"].value_counts())

   Question_ID                                           Question  \
0            0  Why is every book I hear about a " NY Times # ...   
1            1  If salt is so bad for cars , why do we use it ...   
2            2  Why do we still have SD TV channels when HD lo...   
3            3  Why has nobody assassinated Kim Jong - un He i...   
4            4  How was airplane technology able to advance so...   

        Source  Group                                           Response  
0  reddit_eli5  Human  ['Basically there are many categories of " Bes...  
1  reddit_eli5  Human  ['salt is good for not dying in car crashes an...  
2  reddit_eli5  Human  ["The way it works is that old TV stations got...  
3  reddit_eli5  Human  ["You ca n't just go around assassinating the ...  
4  reddit_eli5  Human  ['Wanting to kill the shit out of Germans driv...  
Group
Human     24322
Old_AI    24322
New_AI     2000
Name: count, dtype: int64
